# Khan Academy — Reading Fluency Tracker

**Goal:** Build a school-facing tracker and an internal consolidated tracker for a
Reading Fluency test (CMF — "Curriculum-based Measurement of Fluency"), from raw
student, test-attempt, and test-result data, plus a school sign-up form.

This notebook is fully self-contained and idempotent — it can be re-run daily
(e.g. as a scheduled job) to refresh the trackers as new data arrives.

**What this notebook does, top to bottom:**
1. Load the three raw JSON extracts into a local SQLite database (`student_details`,
   `cmf_input`, `cmf_metrics`).
2. Load the `FormResponse` sheet from the provided Excel workbook into a
   `form_response` table.
3. Build a **consolidated school tracker** (one row per school) — this is what the
   internal Khan Academy team would use to monitor all schools at once.
4. Build **school-wise reports** (overall + grade-wise breakdown) for
   `SCH_134065` and `SCH_134141` — this is the format an individual school would see.
5. Wrap the whole pipeline in a single `refresh_trackers()` function so it can be
   scheduled to run daily against new files with no manual steps.

**Data files expected in the same folder as this notebook** (as provided):
`StudentDetails.json`, `CMFInput.json`, `CMFMetrics.json`,
`Senior_Data_Analyst_Assignment.xlsx`. No internet access or external services are
required — everything runs locally with the Python standard library + pandas +
SQLite.


## Assumptions & data-treatment decisions

Since this is a take-home based on a real-ish sample export, the raw data has a
few quirks. Rather than silently "fixing" them, the assumptions below are made
explicit so the impact is auditable:

1. **`student_details` has duplicate `id`s (176 of 1,683 rows, 1,439 unique).**
   Some duplicates are exact repeats; others show the same `id`/contact/email but a
   different `name` and even a different `school_id` (looks like re-registration or
   test-data noise). *Assumption:* `id` is treated as the true primary key and the
   **last** record for a given `id` in the file wins (`INSERT OR REPLACE`, in file
   order). This mirrors how an upsert-based nightly sync would behave.
2. **`cmf_input.school_id` doesn't always match the school on the student's own
   record** (267 of 1,382 rows, largely driven by the duplicate/re-registration
   issue above). *Assumption:* the **school_id recorded on the test attempt itself**
   (`cmf_input.school_id`) is treated as the source of truth for "which school does
   this attempt belong to," since that's the record created at the moment the test
   was taken. Grade-level cuts still join back to `student_details` for `grade`.
3. **`cmf_metrics` only exists for `status = 'Processed'` attempts** (1,144
   Processed rows in `cmf_input` = 1,144 rows in `cmf_metrics`, a clean 1:1 match).
   `Failed` attempts (238) have no metrics, which makes sense — a failed attempt was
   never scored. All average-score metrics are therefore computed over **Processed**
   attempts only.
4. **"cmf submitted" vs "cmf successful."** `cmf submitted` = every attempt logged
   in `cmf_input` for a school, regardless of outcome. `cmf successful` = the subset
   with `status = 'Processed'` (i.e., a valid, scored attempt). This distinguishes
   "did they try" from "did we get a usable result."
5. **`Total Student`** in the trackers is taken from `no_of_students` on the
   school's sign-up form (`FormResponse`), since that reflects the school's own
   reported enrolment. **`Registered Users`** is the count of distinct students in
   `student_details` for that school — i.e., how many of the school's students have
   actually been onboarded to the app, which is naturally ≤ `Total Student`.
6. **Forms without matching JSON data (or vice versa)** are kept via `LEFT JOIN`s
   from `form_response` so every school that signed up appears in the tracker even
   if it has zero activity so far, with `0`/blank metrics.
7. All numeric average metrics are rounded to 2 decimal places for readability;
   underlying tables retain full precision.


## 1. Setup

In [1]:
import json
import os
import sqlite3
from datetime import datetime

import pandas as pd

DB_PATH = "student_academy_fluency.db"
DATA_DIR = "."   # folder containing the JSON files and the Excel workbook

STUDENT_DETAILS_FILE = os.path.join(DATA_DIR, "StudentDetails.json")
CMF_INPUT_FILE       = os.path.join(DATA_DIR, "CMFInput.json")
CMF_METRICS_FILE     = os.path.join(DATA_DIR, "CMFMetrics.json")
WORKBOOK_FILE        = os.path.join(DATA_DIR, "Senior_Data_Analyst_Assignment.xlsx")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


## 2. Initialize the local database & define the schema

Four tables are created:

- `student_details` — one row per student (from `StudentDetails.json`)
- `cmf_input` — one row per test *attempt* (from `CMFInput.json`)
- `cmf_metrics` — one row per *scored* attempt (from `CMFMetrics.json`)
- `form_response` — one row per school sign-up (from the Excel `FormResponse` sheet)

Written in plain, portable SQL (`CREATE TABLE`, standard types) so this schema can
be dropped into Postgres or any other RDBMS with little to no change — SQLite is
used here purely so the notebook is self-contained and needs no external DB server.


In [2]:
def init_db(db_path=DB_PATH, drop_existing=True):
    """Create (or recreate) the local SQLite database and its schema."""
    if drop_existing and os.path.exists(db_path):
        os.remove(db_path)
    conn = sqlite3.connect(db_path)
    conn.executescript('''
        CREATE TABLE IF NOT EXISTS student_details (
            id              TEXT PRIMARY KEY,
            name            TEXT,
            contact_number  TEXT,
            email_id        TEXT,
            school_id       TEXT,
            grade           TEXT
        );

        CREATE TABLE IF NOT EXISTS cmf_input (
            id              TEXT PRIMARY KEY,
            child_id        TEXT,
            school_id       TEXT,
            submitted_at    TEXT,
            status          TEXT
        );

        CREATE TABLE IF NOT EXISTS cmf_metrics (
            id              TEXT PRIMARY KEY,
            input_id        TEXT UNIQUE,
            processed_date  TEXT,
            wpm             REAL,
            wcpm            REAL,
            pronunciation   REAL,
            fluency         REAL,
            noise           REAL
        );

        CREATE TABLE IF NOT EXISTS form_response (
            school_id             TEXT PRIMARY KEY,
            school_name           TEXT,
            city                  TEXT,
            state                 TEXT,
            contact_number        TEXT,
            email_id              TEXT,
            no_of_students        INTEGER,
            form_submission_date  TEXT
        );

        CREATE INDEX IF NOT EXISTS idx_student_school ON student_details(school_id);
        CREATE INDEX IF NOT EXISTS idx_cmfinput_school ON cmf_input(school_id);
        CREATE INDEX IF NOT EXISTS idx_cmfinput_child   ON cmf_input(child_id);
    ''')
    conn.commit()
    return conn

conn = init_db()
print("Database initialized at", os.path.abspath(DB_PATH))


Database initialized at C:\Users\skumar37\Downloads\khan_acadmey\khan_academy_fluency.db


## 3. Ingest the JSON files into the database

`INSERT OR REPLACE` is used throughout (an *upsert*) keyed on each table's primary
key. This is what makes the load idempotent and daily-run-safe: re-running the
notebook against an updated export simply refreshes rows that changed and adds new
ones — nothing has to be deleted/rebuilt from scratch.


In [3]:
def load_json(path):
    with open(path, "r") as f:
        return json.load(f)

def ingest_student_details(conn, path=STUDENT_DETAILS_FILE):
    records = load_json(path)
    conn.executemany(
        """INSERT OR REPLACE INTO student_details
               (id, name, contact_number, email_id, school_id, grade)
           VALUES (:id, :name, :contact_number, :email_id, :school_id, :grade)""",
        [{**r, "contact_number": str(r["contact_number"])} for r in records],
    )
    conn.commit()
    return len(records)

def ingest_cmf_input(conn, path=CMF_INPUT_FILE):
    records = load_json(path)
    conn.executemany(
        """INSERT OR REPLACE INTO cmf_input
               (id, child_id, school_id, submitted_at, status)
           VALUES (:id, :child_id, :school_id, :submitted_at, :status)""",
        records,
    )
    conn.commit()
    return len(records)

def ingest_cmf_metrics(conn, path=CMF_METRICS_FILE):
    records = load_json(path)
    conn.executemany(
        """INSERT OR REPLACE INTO cmf_metrics
               (id, input_id, processed_date, wpm, wcpm, pronunciation, fluency, noise)
           VALUES (:id, :input_id, :processed_date, :wpm, :wcpm, :pronunciation, :fluency, :noise)""",
        records,
    )
    conn.commit()
    return len(records)

n_students = ingest_student_details(conn)
n_inputs   = ingest_cmf_input(conn)
n_metrics  = ingest_cmf_metrics(conn)

print(f"StudentDetails.json : {n_students:>5} records read")
print(f"CMFInput.json       : {n_inputs:>5} records read")
print(f"CMFMetrics.json     : {n_metrics:>5} records read")
print()
for t in ["student_details", "cmf_input", "cmf_metrics"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  -> {t:<16} {n:>5} rows after upsert")


StudentDetails.json :  1683 records read
CMFInput.json       :  1382 records read
CMFMetrics.json     :  1144 records read

  -> student_details   1439 rows after upsert
  -> cmf_input         1382 rows after upsert
  -> cmf_metrics       1144 rows after upsert


## 4. Extract `FormResponse` from the Excel workbook and load `form_response`

Each school appears once in `FormResponse`; the sheet is read with pandas and
upserted into the `form_response` table the same way as the JSON sources.


In [4]:
def ingest_form_response(conn, path=WORKBOOK_FILE, sheet_name="FormResponse"):
    df = pd.read_excel(path, sheet_name=sheet_name)
    df = df.dropna(subset=["school_id"]).copy()

    df["form_submission_date"] = df["form_submission_date"].astype(str)
    df["contact_number"] = df["contact_number"].apply(
        lambda x: str(int(x)) if pd.notna(x) else None
    )
    df["no_of_students"] = pd.to_numeric(df["no_of_students"], errors="coerce")

    rows = df[[
        "school_id", "school_name", "city", "state",
        "contact_number", "email_id", "no_of_students", "form_submission_date",
    ]].to_dict(orient="records")

    conn.executemany(
        """INSERT OR REPLACE INTO form_response
               (school_id, school_name, city, state, contact_number, email_id,
                no_of_students, form_submission_date)
           VALUES (:school_id, :school_name, :city, :state, :contact_number,
                   :email_id, :no_of_students, :form_submission_date)""",
        rows,
    )
    conn.commit()
    return len(rows)

n_forms = ingest_form_response(conn)
print(f"FormResponse sheet  : {n_forms:>5} school sign-ups loaded into form_response")

pd.read_sql_query("SELECT * FROM form_response LIMIT 5", conn)


FormResponse sheet  :    37 school sign-ups loaded into form_response


,school_id,school_name,city,state,contact_number,email_id,no_of_students,form_submission_date
0,SCH_134213,School Name 134213,Delhi,Delhi,911100222445,test_21459@test.com,1200,2021-03-30 11:09:57
1,SCH_134040,School Name 134040,Amravati,Maharashtra,911100222446,test_21460@test.com,150,2021-03-30 11:48:31
2,SCH_134085,School Name 134085,Greater Noida,Uttar Pradesh,911100222447,test_21461@test.com,162,2021-03-31 03:59:03
3,SCH_134047,School Name 134047,Ghaziabad,Uttar Pradesh,911100222448,test_21462@test.com,150,2021-03-31 04:22:44
4,SCH_134018,School Name 134018,Pune,Maharashtra,911100222449,test_21463@test.com,750,2021-03-31 05:19:28


## 5. Consolidated school tracker (internal Khan Academy view)

One row per school, following the columns from the *Tracker format* sheet:

`school_id | school_name | Total Student | Registered Users | cmf submitted | cmf successful | avg. wpm | avg. wcpm | avg. pronunciation | avg. fluency`

Built as a single portable SQL query (CTEs + LEFT JOINs), so it can be pointed at
Postgres unchanged.


In [5]:
CONSOLIDATED_TRACKER_SQL = """
WITH registered AS (
    SELECT school_id, COUNT(DISTINCT id) AS registered_users
    FROM student_details
    GROUP BY school_id
),
submitted AS (
    SELECT school_id,
           COUNT(*) AS cmf_submitted,
           SUM(CASE WHEN status = 'Processed' THEN 1 ELSE 0 END) AS cmf_successful
    FROM cmf_input
    GROUP BY school_id
),
metrics AS (
    SELECT ci.school_id,
           AVG(cm.wpm)           AS avg_wpm,
           AVG(cm.wcpm)          AS avg_wcpm,
           AVG(cm.pronunciation) AS avg_pronunciation,
           AVG(cm.fluency)       AS avg_fluency
    FROM cmf_input ci
    JOIN cmf_metrics cm ON cm.input_id = ci.id
    WHERE ci.status = 'Processed'
    GROUP BY ci.school_id
)
SELECT
    fr.school_id                                AS school_id,
    fr.school_name                               AS school_name,
    fr.no_of_students                            AS "Total Student",
    COALESCE(r.registered_users, 0)              AS "Registered Users",
    COALESCE(s.cmf_submitted, 0)                 AS "cmf submitted",
    COALESCE(s.cmf_successful, 0)                AS "cmf successful",
    ROUND(m.avg_wpm, 2)                          AS "avg. wpm",
    ROUND(m.avg_wcpm, 2)                         AS "avg. wcpm",
    ROUND(m.avg_pronunciation, 2)                AS "avg. pronunciation",
    ROUND(m.avg_fluency, 2)                      AS "avg. fluency"
FROM form_response fr
LEFT JOIN registered r ON r.school_id = fr.school_id
LEFT JOIN submitted  s ON s.school_id = fr.school_id
LEFT JOIN metrics    m ON m.school_id = fr.school_id
ORDER BY fr.school_id;
"""

def build_consolidated_tracker(conn):
    return pd.read_sql_query(CONSOLIDATED_TRACKER_SQL, conn)

consolidated_tracker = build_consolidated_tracker(conn)
print(f"Consolidated tracker: {len(consolidated_tracker)} schools")
consolidated_tracker


Consolidated tracker: 37 schools


,school_id,school_name,Total Student,Registered Users,cmf submitted,cmf successful,avg. wpm,avg. wcpm,avg. pronunciation,avg. fluency
0,SCH_134011,School Name 134011,480,41,40,35,105.15,84.43,0.72,0.79
1,SCH_134014,School Name 134014,400,35,28,22,106.76,86.48,0.72,0.81
2,SCH_134016,School Name 134016,50,4,3,2,95.60,69.10,0.68,0.61
3,SCH_134018,School Name 134018,750,68,52,44,105.30,85.55,0.76,0.80
4,SCH_134023,School Name 134023,300,27,26,22,105.71,86.93,0.75,0.81
5,SCH_134026,School Name 134026,1000,83,82,68,101.18,79.36,0.70,0.77
6,SCH_134028,School Name 134028,500,40,51,40,106.79,88.57,0.78,0.81
7,SCH_134031,School Name 134031,200,17,17,15,109.33,87.41,0.72,0.81
8,SCH_134040,School Name 134040,150,13,13,10,117.71,102.93,0.82,0.84
9,SCH_134047,School Name 134047,150,13,13,11,89.11,64.29,0.64,0.77


## 6. School-wise report (school-facing view)

Follows the *School Wise Report* sheet: a **Schoolwise report** block (same
columns as the consolidated tracker, but for a single school) plus a **Grade Wise
report** block breaking the same metrics down by `Grade 1`–`Grade 4`.

As noted in the assumptions, a test attempt is attributed to the school recorded
on `cmf_input.school_id` (not the student's own `school_id`, which can be stale for
the small number of duplicated student records), while `grade` is pulled from
`student_details` via `child_id`.


In [6]:
SCHOOL_SUMMARY_SQL = """
WITH registered AS (
    SELECT school_id, COUNT(DISTINCT id) AS registered_users
    FROM student_details WHERE school_id = ?
    GROUP BY school_id
),
submitted AS (
    SELECT school_id,
           COUNT(*) AS cmf_submitted,
           SUM(CASE WHEN status = 'Processed' THEN 1 ELSE 0 END) AS cmf_successful
    FROM cmf_input WHERE school_id = ?
    GROUP BY school_id
),
metrics AS (
    SELECT ci.school_id,
           AVG(cm.wpm) AS avg_wpm, AVG(cm.wcpm) AS avg_wcpm,
           AVG(cm.pronunciation) AS avg_pronunciation, AVG(cm.fluency) AS avg_fluency
    FROM cmf_input ci
    JOIN cmf_metrics cm ON cm.input_id = ci.id
    WHERE ci.status = 'Processed' AND ci.school_id = ?
    GROUP BY ci.school_id
)
SELECT
    fr.school_id                    AS "School ID",
    fr.school_name                  AS "School Name",
    fr.no_of_students                AS "Total Student",
    COALESCE(r.registered_users, 0)  AS "Registered Users",
    COALESCE(s.cmf_submitted, 0)     AS "cmf submitted",
    COALESCE(s.cmf_successful, 0)    AS "cmf successful",
    ROUND(m.avg_wpm, 2)              AS "avg. wpm",
    ROUND(m.avg_wcpm, 2)             AS "avg. wcpm",
    ROUND(m.avg_pronunciation, 2)    AS "avg. pronunciation",
    ROUND(m.avg_fluency, 2)          AS "avg. fluency"
FROM form_response fr
LEFT JOIN registered r ON r.school_id = fr.school_id
LEFT JOIN submitted  s ON s.school_id = fr.school_id
LEFT JOIN metrics    m ON m.school_id = fr.school_id
WHERE fr.school_id = ?;
"""

GRADE_WISE_SQL = """
WITH grade_students AS (
    SELECT school_id, id AS student_id, grade
    FROM student_details WHERE school_id = ?
),
grade_registered AS (
    SELECT grade, COUNT(DISTINCT student_id) AS registered_users
    FROM grade_students GROUP BY grade
),
grade_submitted AS (
    SELECT gs.grade,
           COUNT(*) AS cmf_submitted,
           SUM(CASE WHEN ci.status = 'Processed' THEN 1 ELSE 0 END) AS cmf_successful
    FROM cmf_input ci
    JOIN grade_students gs ON gs.student_id = ci.child_id
    WHERE ci.school_id = ?
    GROUP BY gs.grade
),
grade_metrics AS (
    SELECT gs.grade,
           AVG(cm.wpm) AS avg_wpm, AVG(cm.wcpm) AS avg_wcpm,
           AVG(cm.pronunciation) AS avg_pronunciation, AVG(cm.fluency) AS avg_fluency
    FROM cmf_input ci
    JOIN grade_students gs ON gs.student_id = ci.child_id
    JOIN cmf_metrics cm ON cm.input_id = ci.id
    WHERE ci.status = 'Processed' AND ci.school_id = ?
    GROUP BY gs.grade
)
SELECT
    gr.grade                          AS "Grade",
    COALESCE(gr.registered_users, 0)  AS "Registered Users",
    COALESCE(gsub.cmf_submitted, 0)   AS "cmf submitted",
    COALESCE(gsub.cmf_successful, 0)  AS "cmf successful",
    ROUND(gm.avg_wpm, 2)              AS "avg. wpm",
    ROUND(gm.avg_wcpm, 2)             AS "avg. wcpm",
    ROUND(gm.avg_pronunciation, 2)    AS "avg. pronunciation",
    ROUND(gm.avg_fluency, 2)          AS "avg. fluency"
FROM grade_registered gr
LEFT JOIN grade_submitted gsub ON gsub.grade = gr.grade
LEFT JOIN grade_metrics   gm   ON gm.grade   = gr.grade
ORDER BY gr.grade;
"""

def build_school_report(conn, school_id):
    '''Return (summary_df, grade_wise_df) for a single school, matching the
    "School Wise Report" sheet layout.'''
    summary = pd.read_sql_query(SCHOOL_SUMMARY_SQL, conn, params=[school_id]*4)
    grade_wise = pd.read_sql_query(GRADE_WISE_SQL, conn, params=[school_id]*3)
    return summary, grade_wise


In [7]:
school_reports = {}
for sid in ["SCH_134065", "SCH_134141"]:
    school_reports[sid] = build_school_report(conn, sid)
    summary, grade_wise = school_reports[sid]
    print(f"===== Schoolwise report: {sid} =====")
    display(summary)
    print(f"===== Grade Wise report: {sid} =====")
    display(grade_wise)
    print()


===== Schoolwise report: SCH_134065 =====


,School ID,School Name,Total Student,Registered Users,cmf submitted,cmf successful,avg. wpm,avg. wcpm,avg. pronunciation,avg. fluency
0,SCH_134065,School Name 134065,1200,109,92,64,104.61,86.29,0.76,0.76


===== Grade Wise report: SCH_134065 =====


,Grade,Registered Users,cmf submitted,cmf successful,avg. wpm,avg. wcpm,avg. pronunciation,avg. fluency
0,Grade 1,22,14,10,62.17,35.87,0.44,0.46
1,Grade 2,28,20,12,103.88,81.48,0.73,0.79
2,Grade 3,25,21,18,99.58,78.77,0.80,0.81
3,Grade 4,34,29,20,134.82,124.05,0.89,0.87



===== Schoolwise report: SCH_134141 =====


,School ID,School Name,Total Student,Registered Users,cmf submitted,cmf successful,avg. wpm,avg. wcpm,avg. pronunciation,avg. fluency
0,SCH_134141,School Name 134141,1500,113,124,102,95.74,71.71,0.66,0.78


===== Grade Wise report: SCH_134141 =====


,Grade,Registered Users,cmf submitted,cmf successful,avg. wpm,avg. wcpm,avg. pronunciation,avg. fluency
0,Grade 1,26,28,23,72.23,40.57,0.53,0.71
1,Grade 2,28,21,18,85.39,55.04,0.58,0.76
2,Grade 3,28,24,15,93.03,68.80,0.67,0.77
3,Grade 4,31,27,25,128.77,115.30,0.86,0.86


## 7. Export the reports (optional convenience step)

Writes the consolidated tracker and both school-wise reports out to a single Excel
workbook mirroring the original *Tracker format* / *School Wise Report* layout,
useful for sharing with non-technical stakeholders.


In [8]:
OUTPUT_XLSX = "fluency_trackers_output.xlsx"

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    consolidated_tracker.to_excel(writer, sheet_name="Overall Tracker", index=False)
    row_cursor = 0
    for sid, (summary, grade_wise) in school_reports.items():
        sheet_name = sid[:31]  # Excel sheet name limit
        summary.to_excel(writer, sheet_name=sheet_name, index=False, startrow=0)
        grade_wise.to_excel(writer, sheet_name=sheet_name, index=False,
                             startrow=len(summary) + 3)

print("Reports exported to", os.path.abspath(OUTPUT_XLSX))


Reports exported to C:\Users\skumar37\Downloads\khan_acadmey\fluency_trackers_output.xlsx


## 8. Daily-refresh entry point

Everything above is wrapped into a single function. Running this cell (or calling
`refresh_trackers()` from a scheduler — cron, Airflow, etc.) re-initializes the DB,
re-ingests the three JSON files + the Excel `FormResponse` sheet, and rebuilds both
the consolidated tracker and the two school-wise reports — all idempotently, so it
is safe to run once a day (or as often as new data lands) with **zero manual
steps**.

In production this would point at whichever daily data drop replaces these static
files (e.g. files landing in cloud storage, or rows landing in a source database) —
only the ingestion functions in Section 3/4 would need to change; the schema and
all SQL/report logic stay exactly the same.


In [9]:
def refresh_trackers(db_path=DB_PATH,
                      student_path=STUDENT_DETAILS_FILE,
                      cmf_input_path=CMF_INPUT_FILE,
                      cmf_metrics_path=CMF_METRICS_FILE,
                      workbook_path=WORKBOOK_FILE,
                      school_ids_for_reports=("SCH_134065", "SCH_134141")):
    '''End-to-end pipeline: ingest -> transform -> report. Safe to schedule daily.'''
    run_conn = init_db(db_path, drop_existing=False)  # upsert, don't blow away history

    ingest_student_details(run_conn, student_path)
    ingest_cmf_input(run_conn, cmf_input_path)
    ingest_cmf_metrics(run_conn, cmf_metrics_path)
    ingest_form_response(run_conn, workbook_path)

    tracker = build_consolidated_tracker(run_conn)
    reports = {sid: build_school_report(run_conn, sid) for sid in school_ids_for_reports}

    run_conn.close()
    return tracker, reports

# Example scheduled run (re-uses the same source files here; in production this
# would point at the latest daily export instead):
tracker_today, reports_today = refresh_trackers()
print(f"[{datetime.now().isoformat(timespec='seconds')}] Daily refresh complete.")
print(f"  Consolidated tracker: {len(tracker_today)} schools")
for sid in reports_today:
    print(f"  School report ready for: {sid}")


[2026-07-01T22:08:13] Daily refresh complete.
  Consolidated tracker: 37 schools
  School report ready for: SCH_134065
  School report ready for: SCH_134141


## Summary

- Local SQLite DB with 4 tables (`student_details`, `cmf_input`, `cmf_metrics`,
  `form_response`), loaded idempotently from the provided JSON/Excel sources.
- **Consolidated school tracker** across all 37 schools for the internal Khan
  Academy team (Section 5).
- **School-wise reports** (overall + grade-wise) for `SCH_134065` and
  `SCH_134141`, matching the *School Wise Report* sheet layout (Section 6).
- A single `refresh_trackers()` function (Section 8) that re-runs the entire
  pipeline idempotently — ready to be scheduled daily against updated source files.
